# H&M 2년 M4 K=1 seed 44 — actual arm 병렬 실행

기존 3-arm 실행과 같은 검토 커밋·설정·체크포인트 경로를 사용해 이 arm 하나만 학습하거나 재개합니다. 다른 두 arm은 각각 별도 Colab 런타임에서 실행할 수 있습니다. 최종 test와 holdout은 만들지 않습니다.

런타임이 끊기면 Drive를 다시 마운트하고 셀을 재실행하세요. 마지막 완료 epoch 다음부터 재개됩니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
import subprocess
import sys

REVIEWED_SHA = '3188b01c360d54e78338eaebdf2a8ecb6a6d52d8'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
REPO_DIR = Path('/content/clv-m2-lightgcn-runner-hm-m4-k1-seed44-actual')

if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'기존 경로가 Git 저장소가 아닙니다: {REPO_DIR}')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '-q', 'origin', REVIEWED_SHA], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-q', '--detach', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('검토된 코드:', actual_sha)


In [ ]:
import json
import torch
import lightgcn_clv_m4_k1_assignment_control_hm2y as hm_screen

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert hm_screen.CODE_VERSION == 'm4-personalized-positive-weight-k1-assignment-control-hm2y-development-screen-v1'
cfg = hm_screen.configure_hm2y_m4_assignment_screen(seed=44, shuffle_seed=44)
summary = hm_screen.preflight_summary(cfg)
assert summary['dataset'] == 'hm'
assert summary['seed'] == 44 and cfg.shuffle_seed == 44
assert summary['staged_replication']['seeds'] == [42, 43, 44]
assert summary['trained_models'] == list(hm_screen.MODEL_IDS)
assert summary['fixed']['negative_count'] == 1
assert summary['fixed']['batch_size'] == 131072
assert summary['fixed']['final_test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
assert summary['checkpointing']['save_after_each_completed_epoch'] is True
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\n중단 후 재실행하면 마지막으로 완료된 epoch 다음부터 이어집니다.')

SELECTED_MODEL_ID = 'm4_personalized_positive_weight_actual_qc_bpr_k1_hm2y'
assert SELECTED_MODEL_ID in hm_screen.MODEL_IDS
print('선택 arm:', SELECTED_MODEL_ID)


In [ ]:
# 기존 3-arm runner와 같은 config hash를 사용하고 선택 arm만 실행합니다.
prepared = hm_screen._prepare(cfg)
spec = next(spec for spec in hm_screen.arm_specifications(prepared) if spec['model_id'] == SELECTED_MODEL_ID)
print(f"\n===== {SELECTED_MODEL_ID} | seed {cfg.seed} | 병렬 단일 arm =====")
arm_result, _ = hm_screen.training._run_arm(hm_screen.arm_prepared(prepared, spec), cfg, spec)
arm_result['m4_assignment'] = spec['m4_assignment']
print('완료:', SELECTED_MODEL_ID, 'epoch', arm_result['final_epoch'])
